In [ ]:
%reload_ext autoreload
%autoreload 2

import os
import sys
from pathlib import Path

project_root = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src" / "global_settings.py").is_file()
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
os.chdir(project_root)
from src.global_settings import settings


In [ ]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import matplotlib.pyplot as plt

from src.activations import ActivationDataset

In [ ]:
raw = pd.read_csv("./data/text_dataset/raw.csv")

print("Loaded Dataset!\ntotal:", len(raw))
print(raw["label"].value_counts())

raw.head()

In [ ]:
grp_md = raw.groupby(["model", "prompt"]).size().reset_index(name="count").to_markdown()

In [ ]:
df = pd.read_csv(str(settings.text_dataset_path))

grp_md = df.groupby(["model", "prompt"]).size().reset_index(name="count").to_markdown()

- 토큰 길이 분포 확인 (max_length를 정하기 위함. 너무 길면 메모리 낭비 심함)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(str(settings.lm_weights_path))

def check_token_length(text):
    tokens = tokenizer(text, return_tensors="pt", truncation=False)
    return len(tokens["input_ids"][0])

df["token_length"] = df["text"].apply(check_token_length)

print("Max token length:", df["token_length"].max())
print("Min token length:", df["token_length"].min())
print("Mean token length:", round(df["token_length"].mean(), 2))

del tokenizer

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(df["token_length"], bins=500, color="skyblue", edgecolor="black")
plt.xlim(0, 2048)
plt.title("Token Length Distribution")
plt.xlabel("Token Length")
plt.ylabel("Frequency")
plt.grid(axis="y", alpha=0.75)
plt.show()

In [ ]:
model_path = str(settings.lm_weights_path)

model = AutoModelForSequenceClassification.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained(model_path)

print(f"Model loaded: {model.config.model_type}")
print(f"Number of Layers: {model.config.num_hidden_layers}")
print(f"Hidden Size: {model.config.hidden_size}")

In [ ]:
model.config

In [ ]:
model

In [ ]:
base_model = model.base_model
layers = base_model.layers
layer_0 = base_model.layers[0]

In [ ]:
layer_0

ModernBertEncoderLayer(
  (attn_norm): Identity()
  (attn): ModernBertAttention(
    (Wqkv): Linear(in_features=768, out_features=2304, bias=False)
    (rotary_emb): ModernBertRotaryEmbedding()
    (Wo): Linear(in_features=768, out_features=768, bias=False)
    (out_drop): Identity()
  )
  (mlp_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  (mlp): ModernBertMLP(
    (Wi): Linear(in_features=768, out_features=2304, bias=False)
    (act): GELUActivation()
    (drop): Dropout(p=0.0, inplace=False)
    (Wo): Linear(in_features=1152, out_features=768, bias=False)
  )
)

### 옵션-A. 요기서 residual stream 추출 (attn_out)

In [ ]:
layer_0.attn

ModernBertAttention(
  (Wqkv): Linear(in_features=768, out_features=2304, bias=False)
  (rotary_emb): ModernBertRotaryEmbedding()
  (Wo): Linear(in_features=768, out_features=768, bias=False)
  (out_drop): Identity()
)

### 옵션-B: 요기서 residual stream 추출 (mlp_out) (*현재 이 방식)

In [ ]:
layer_0.mlp

ModernBertMLP(
  (Wi): Linear(in_features=768, out_features=2304, bias=False)
  (act): GELUActivation()
  (drop): Dropout(p=0.0, inplace=False)
  (Wo): Linear(in_features=1152, out_features=768, bias=False)
)

# Residual Stream 텐서 확인

In [ ]:
activation_dataset = ActivationDataset("./data/post_mlp_residual_stream/layer_1")

In [ ]:
activation_dataset[0], activation_dataset[0].shape

(tensor([[ -8.4766, -11.9297,  -7.4766,  ...,  -8.7891,  -8.6797,  -8.4688],
         [ -8.8281,  -5.2930,  -6.4375,  ...,  -6.1211,  -9.4609,  -6.1094],
         [ -6.6992,  -3.6934,  -5.2344,  ...,  -5.5625,  -7.1523,  -7.0039],
         ...,
         [ -9.0938, -11.0938,  -6.8750,  ...,  -9.7031,  -6.9375,  -8.1484],
         [ -9.1172, -11.1875,  -6.8047,  ...,  -9.5312,  -6.8516,  -8.2812],
         [ -9.5781, -11.3906,  -6.0625,  ...,  -9.2344,  -7.2656,  -9.6172]],
        dtype=torch.float16),
 torch.Size([131072, 768]))

아래는 activation 데이터 추출 과정을 요약한 데모 코드

In [ ]:
base_model = model.base_model
activations = []

def get_activation_hook(module, input, output):
    activations.append(input[0].detach().cpu())

target_layer_idx = 10


target_module = base_model.layers[target_layer_idx]
handle = target_module.register_forward_hook(get_activation_hook)

sample_text = df["text"].tolist()[0]
inputs = tokenizer(sample_text, return_tensors="pt")

model.eval()
with torch.no_grad():
    outputs = model(**inputs)

handle.remove()

residual_stream = activations[0]
print(f"Tensor Shape: {residual_stream.shape}")
print(f"(Batch Size, Sequence Length, Hidden Dimension)")

Tensor Shape: torch.Size([1, 206, 768])

(Batch Size, Sequence Length, Hidden Dimension)